# Probability Distributions

## Section 1: Generating Data from a Specific Distribution

In [1]:
# Exercise 1: Uniform Generator
import numpy as np
def uniform_generator(a, b, num_samples=100):
    np.random.seed(42)
    
    array = np.random.uniform(a, b, num_samples)
    
    return array

In [2]:
print(f"6 randomly generated values between 0 and 1:\n{np.array2string(uniform_generator(0, 1, num_samples=6), precision=3)}\n")
print(f"3 randomly generated values between 20 and 55:\n{np.array2string(uniform_generator(20, 55, num_samples=3), precision=3)}\n")
print(f"1 randomly generated value between 0 and 100:\n{np.array2string(uniform_generator(0, 100, num_samples=1), precision=3)}")

6 randomly generated values between 0 and 1:
[0.375 0.951 0.732 0.599 0.156 0.156]

3 randomly generated values between 20 and 55:
[33.109 53.275 45.62 ]

1 randomly generated value between 0 and 100:
[37.454]


In [3]:
# Exercise 2: Gaussian Generator

# Inverse CDF - Gaussian
def inverse_cdf_gaussian(y, mu, sigma):
    x = sigma * np.sqrt(2) * erfinv(2 * y - 1) + mu
    return x

In [10]:
def gaussian_generator(mu, sigma, num_samples):
    ### START CODE HERE ###

    # Generate an array with num_samples elements that distribute uniformally between 0 and 1
    u = uniform_generator(0,1,num_samples)

    # Use the uniform-distributed sample to generate Gaussian-distributed data
    # Hint: You need to sample from the inverse of the CDF of the distribution you are generating
    array = inverse_cdf_gaussian(u, mu, sigma)
    ### END CODE HERE ###

    return array

## Section 2: Building a Dog Breed Classifier using Naive Bayes

### Section 2.1: Generating the Dataset

In [4]:
FEATURES = ["height", "weight", "bark_days", "ear_head_ratio"]

In [5]:
# learn how to create a dataclass
'''
@dataclass
class my_data_class:
    my_var: str
        
foo = my_data_class(my_var="Hello World")
'''
from dataclasses import dataclass

@dataclass
class params_gaussian:
    mu: float
    sigma: float
        
    def __repr__(self):
        return f"params_gaussian(mu={self.mu:.3f}, sigma={self.sigma:.3f})"


@dataclass
class params_binomial:
    n: int
    p: float
        
    def __repr__(self):
        return f"params_binomial(n={self.n:.3f}, p={self.p:.3f})"


@dataclass
class params_uniform:
    a: int
    b: int
        
    def __repr__(self):
        return f"params_uniform(a={self.a:.3f}, b={self.b:.3f})"

In [6]:
breed_params = {
    0: {
        "height": params_gaussian(mu=35, sigma=1.5),
        "weight": params_gaussian(mu=20, sigma=1),
        "bark_days": params_binomial(n=30, p=0.8),
        "ear_head_ratio": params_uniform(a=0.6, b=0.1)
    },
    
    1: {
        "height": params_gaussian(mu=30, sigma=2),
        "weight": params_gaussian(mu=25, sigma=5),
        "bark_days": params_binomial(n=30, p=0.5),
        "ear_head_ratio": params_uniform(a=0.2, b=0.5)
    },
    
    2: {
        "height": params_gaussian(mu=40, sigma=3.5),
        "weight": params_gaussian(mu=32, sigma=3),
        "bark_days": params_binomial(n=30, p=0.3),
        "ear_head_ratio": params_uniform(a=0.1, b=0.3)
    }
    
}

In [8]:
def generate_data_for_breed(breed, features, n_samples, params):
    df = pd.DataFrame()
    
    for feature in features:
        if feature=="height" or feature=="weight":
            df[feature] = gaussian_generator(params[breed][feature].mu, params[breed][feature].sigma, n_samples)
        elif feature== "bark_days":     
            df[feature] = binomial_generator(params[breed][feature].n, params[breed][feature].p, n_samples)
        elif feature== "ear_head_ratio":                             
            df[feature] = uniform_generator(params[breed][feature].a, params[breed][feature].b, n_samples)    
    
    df["breed"] = breed
    
    return df

In [11]:
import pandas as pd
# Generate data for each breed
df_0 = generate_data_for_breed(breed=0, features=FEATURES, n_samples=1200, params=breed_params)
df_1 = generate_data_for_breed(breed=1, features=FEATURES, n_samples=1350, params=breed_params)
df_2 = generate_data_for_breed(breed=2, features=FEATURES, n_samples=900, params=breed_params)


# concatenate all breeds into a single dataframe
df_all_breads = pd.concat([df_0, df_1, df_2].reset_index(drop=True))

NameError: name 'erfinv' is not defined

In [12]:
# real-life data won't have this nice behavior

## Section 3 - Spam Detector

In [2]:
# train a naive bayes classifier

# detect spam from ham
# the features are categorical


# loading the dataset
import pandas as pd
emails = pd.read_csv("emails.csv")

In [3]:
def process_email(text):
    text = text.lower()
    return list(set(text.split()))

In [4]:
# 创建一个新的column
# 对某一列应用某一个函数 apply
emails['words'] = emails['text'].apply(process_email)

In [6]:
emails.head(5)

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[changes, in, make, ieader, budget, list, sure..."
1,Subject: the stock trading gunslinger fanny i...,1,"[sapling, ramble, tanzania, mcdougall, like, e..."
2,Subject: unbelievable new homes made easy im ...,1,"[pittman, visit, in, unbelievable, post, your,..."
3,Subject: 4 color printing special request add...,1,"[printing, /, 338, an, irwindale, our, e, prin..."
4,"Subject: do not have money , get software cds ...",1,"[old, me, ', be, finish, grow, money, d, from,..."


In [7]:
def word_freq_per_class(df):
    word_freq_dict = {}
    for _, row in df.iterrows():
        # Iterate over the words in each email
        for word in row['words']:
            # Check if word doesn't exist within the dictionary
            if word not in word_freq_dict:
                # If word doesn't exist, initialize the count at 0
                word_freq_dict[word] = {'spam': 0, 'ham': 0}
            
            # Check if the email was spam
            else:
                if row['spam']==0:
                    # If ham then add 1 to the count of ham
                    word_freq_dict[word]['ham'] += 1
                else: 
                    # If spam then add 1 to the count of spam
                    word_freq_dict[word]['spam'] += 1
                    
    ### END CODE HERE ###

    return word_freq_dict


In [8]:
word_freq = word_freq_per_class(emails)
print(f"Frequency in both classes for word 'lottery': {word_freq['lottery']}\n")
print(f"Frequency in both classes for word 'sale': {word_freq['sale']}\n")

try:
    word_freq['asdfg']
except KeyError:
    print("Word 'asdfg' not in corpus")

Frequency in both classes for word 'lottery': {'spam': 7, 'ham': 0}

Frequency in both classes for word 'sale': {'spam': 37, 'ham': 41}

Word 'asdfg' not in corpus


In [9]:
# calculate frequency of words
def word_freq_per_class(df):
    
    word_freq_dict = {}
    
    for _, row in df.iterrows():
        # Iterate over the words in each email
        for word in row['words']:
            # Check if word doesn't exist within the dictionary
            if word not in word_freq_dict:
                # If word doesn't exist, initialize the count at 0
                word_freq_dict[word] = {'spam': 0, 'ham': 0}
            
            # Check if the email was spam
            if row['spam']==0:
                    # If ham then add 1 to the count of ham
                    word_freq_dict[word]['ham'] += 1
            elif row['spam']==1: 
                    # If spam then add 1 to the count of spam
                    word_freq_dict[word]['spam'] += 1
                    
    ### END CODE HERE ###

    return word_freq_dict

In [10]:
# 统计每个单词在spam和ham中的次数
word_freq = word_freq_per_class(emails)
print(f"Frequency in both classes for word 'lottery': {word_freq['lottery']}\n")
print(f"Frequency in both classes for word 'sale': {word_freq['sale']}\n")

try:
    word_freq['asdfg']
except KeyError:
    print("Word 'asdfg' not in corpus")

Frequency in both classes for word 'lottery': {'spam': 8, 'ham': 0}

Frequency in both classes for word 'sale': {'spam': 38, 'ham': 41}

Word 'asdfg' not in corpus


In [11]:
# frequency of classes
# pandas 就是好用
def class_frequencies(df):
    class_freq_dict = { 
        "spam": len(df[df['spam'] == 1]),
        "ham": len(df[df['spam'] == 0])
    } 
    
    return class_freq_dict

In [12]:
# Test your function

class_freq = class_frequencies(emails[:10])
print(f"Small dataset:\n\nThe frequencies for each class are {class_freq}\n")
print(f"The proportion of spam in the dataset is: {100*class_freq['spam']/len(emails[:10]):.2f}%\n")
print(f"The proportion of ham in the dataset is: {100*class_freq['ham']/len(emails[:10]):.2f}%\n")

class_freq = class_frequencies(emails)
print(f"\nFull dataset:\n\nThe frequencies for each class are {class_freq}\n")
print(f"The proportion of spam in the dataset is: {100*class_freq['spam']/len(emails):.2f}%\n")
print(f"The proportion of ham in the dataset is: {100*class_freq['ham']/len(emails):.2f}%")

Small dataset:

The frequencies for each class are {'spam': 10, 'ham': 0}

The proportion of spam in the dataset is: 100.00%

The proportion of ham in the dataset is: 0.00%


Full dataset:

The frequencies for each class are {'spam': 1368, 'ham': 4360}

The proportion of spam in the dataset is: 23.88%

The proportion of ham in the dataset is: 76.12%


In [13]:
# naive bayes for categorical features

def naive_bayes_classifier(text, word_freq=word_freq,class_freq=class_freq):
    text = text.lower()
    words = set(text.split())
    
    cumulative_product_spam = 1.0
    cumulative_product_ham = 1.0
    
    # 全部词汇的set
    for word in words:
        if word in word_freq:
            cumulative_product_spam *= word_freq[word]['spam'] / class_freq['spam']
            cumulative_product_ham *= word_freq[word]['ham'] / class_freq['ham']
    
    likelihood_word_given_spam = cumulative_product_spam * (class_freq['spam'] / (class_freq['spam'] + class_freq['ham']))
    
    # Calculate the likelihood of the words appearing in the email given that it is ham
    likelihood_word_given_ham = cumulative_product_ham * (class_freq['ham'] / (class_freq['spam'] + class_freq['ham']))
    
    # Calculate the posterior probability of the email being spam given that the words appear in the email (the probability of being a spam given the email content)
    prob_spam = likelihood_word_given_spam / (likelihood_word_given_spam + likelihood_word_given_ham)
    
    
    return prob_spam

In [14]:
print(class_freq['spam'] + class_freq['ham'])

5728


In [15]:
msg = "enter the lottery to win three million dollars"
print(f"Probability of spam for email '{msg}': {100*naive_bayes_classifier(msg):.2f}%\n")

msg = "meet me at the lobby of the hotel at nine am"
print(f"Probability of spam for email '{msg}': {100*naive_bayes_classifier(msg):.2f}%\n")

msg = "9898 asjfkjfdj"
print(f"Probability of spam for email '{msg}': {100*naive_bayes_classifier(msg):.2f}%\n")

Probability of spam for email 'enter the lottery to win three million dollars': 100.00%

Probability of spam for email 'meet me at the lobby of the hotel at nine am': 0.00%

Probability of spam for email '9898 asjfkjfdj': 23.88%

